In [1]:
!pip -q install -U "transformers==4.45.2" "accelerate==0.34.2" "peft==0.12.0" \
  "evaluate==0.4.3" "scikit-learn==1.5.2" "datasets==2.21.0" "huggingface_hub==0.26.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 22.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec=

In [2]:
import os, gc, time, math, json, random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer,
    set_seed
)
from peft import PromptTuningConfig, TaskType, get_peft_model
from sklearn.metrics import f1_score, accuracy_score

# ============ Unified config ============
SEED = 42
set_seed(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MAX_INPUT_LEN = 256               # unified
MAX_TARGET_LEN_LABEL = 4          # "TRUE"/"FAKE" very short
MAX_TARGET_LEN_COT = 48           # unified for CoT target (reason + final answer)
GEN_MAX_NEW_TOKENS = 48           # unified decoding budget (must be >= CoT end)
NUM_EPOCHS = 3
EVAL_BATCH = 16                   # keep stable
TRAIN_BATCH = 8                   # safe on T4 for T5-small/BART-base full finetune
GRAD_ACCUM = 4                    # effective batch = 8*4 = 32
LR_FULL = 2e-5
LR_PROMPT = 5e-3
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "GPU:", torch.cuda.get_device_name(0) if DEVICE=="cuda" else None)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

DEVICE: cuda GPU: Tesla T4


In [3]:
import urllib.request

base = "https://raw.githubusercontent.com/tfs4/liar_dataset/master/"
for fn in ["train.tsv", "valid.tsv", "test.tsv"]:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(base + fn, fn)
        print("downloaded", fn)

# LIAR TSV format: see repo README (cols include label and statement)  [oai_citation:2‡GitHub](https://github.com/tfs4/liar_dataset)

downloaded train.tsv
downloaded valid.tsv
downloaded test.tsv


In [4]:
import csv

COL_ID = 0
COL_LABEL = 1
COL_STATEMENT = 2

def read_liar_tsv(path):
    df = pd.read_csv(
        path, sep="\t", header=None, quoting=csv.QUOTE_NONE, engine="python",
        on_bad_lines="skip"
    )
    df = df[[COL_ID, COL_LABEL, COL_STATEMENT]].copy()
    df.columns = ["id", "label6", "statement"]
    return df

train_df = read_liar_tsv("train.tsv")
valid_df = read_liar_tsv("valid.tsv")
test_df  = read_liar_tsv("test.tsv")

# 6-class -> binary (common choice)
POS = {"true", "mostly-true", "half-true"}
def to_binary(lbl):
    lbl = str(lbl).strip().lower()
    return "TRUE" if lbl in POS else "FAKE"

for df in [train_df, valid_df, test_df]:
    df["label"] = df["label6"].apply(to_binary)

print(train_df.head())
print(train_df["label"].value_counts(), "\n", valid_df["label"].value_counts())

           id       label6                                          statement  \
0   2635.json        false  Says the Annies List political group supports ...   
1  10540.json    half-true  When did the decline of coal start? It started...   
2    324.json  mostly-true  Hillary Clinton agrees with John McCain "by vo...   
3   1123.json        false  Health care reform legislation is likely to ma...   
4   9028.json    half-true  The economic turnaround started at the end of ...   

  label  
0  FAKE  
1  TRUE  
2  TRUE  
3  FAKE  
4  TRUE  
label
TRUE    5772
FAKE    4497
Name: count, dtype: int64 
 label
TRUE    668
FAKE    616
Name: count, dtype: int64


In [5]:
INSTR_TEMPLATE = (
    "Task: Classify the following statement as TRUE or FAKE.\n"
    "Statement: {statement}\n"
    "Answer:"
)

# CoT: 要求“短理由 + 最终答案”，并统一成可解析格式
COT_TEMPLATE = (
    "Task: Classify the following statement as TRUE or FAKE.\n"
    "Give ONE brief reason, then output the final label.\n"
    "Statement: {statement}\n"
    "Format:\n"
    "Reason: <one short sentence>\n"
    "Final answer: <TRUE/FAKE>\n"
)

In [6]:
import re

def parse_label(text: str):
    if text is None:
        return None
    t = text.strip().upper()
    # prefer explicit pattern
    m = re.search(r"FINAL\s*ANSWER\s*:\s*(TRUE|FAKE)", t)
    if m:
        return m.group(1)
    # fallback: first TRUE/FAKE token
    m = re.search(r"\b(TRUE|FAKE)\b", t)
    return m.group(1) if m else None

def compute_metrics_from_text(pred_texts, gold_labels):
    preds = [parse_label(x) for x in pred_texts]
    gold = [g.strip().upper() for g in gold_labels]

    # filter unparsable outputs
    keep = [i for i,(p,g) in enumerate(zip(preds,gold)) if p in ("TRUE","FAKE") and g in ("TRUE","FAKE")]
    if len(keep) == 0:
        return {"acc": 0.0, "macro_f1": 0.0, "valid_rate": 0.0}

    preds_k = [preds[i] for i in keep]
    gold_k  = [gold[i] for i in keep]

    acc = accuracy_score(gold_k, preds_k)
    f1  = f1_score(gold_k, preds_k, average="macro")
    return {"acc": float(acc), "macro_f1": float(f1), "valid_rate": len(keep)/len(gold)}

@torch.inference_mode()
def timed_generate(model, tokenizer, texts, batch_size=8):
    model.eval()
    t0 = time.time()
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
        gen = model.generate(
            **enc,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            num_beams=1,
            do_sample=False
        )
        outs.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    if model.device.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.time()
    return outs, (t1 - t0) / max(1, len(texts))  # seconds per example

In [7]:
ds = DatasetDict({
    "train": Dataset.from_pandas(train_df[["statement","label"]], preserve_index=False),
    "validation": Dataset.from_pandas(valid_df[["statement","label"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["statement","label"]], preserve_index=False),
})

def make_view(example, mode="sft"):
    st = example["statement"]
    if mode == "cot":
        inp = COT_TEMPLATE.format(statement=st)
        # 训练目标里包含短理由 + Final answer
        tgt = f"Reason: Because it is consistent with reliable facts.\nFinal answer: {example['label']}"
    else:
        inp = INSTR_TEMPLATE.format(statement=st)
        tgt = example["label"]
    return {"input_text": inp, "target_text": tgt}

views = {
    "sft": ds.map(lambda x: make_view(x, "sft")),
    "cot": ds.map(lambda x: make_view(x, "cot")),
}

Map:   0%|          | 0/10269 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1283 [00:00<?, ? examples/s]

Map:   0%|          | 0/10269 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1283 [00:00<?, ? examples/s]

In [8]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def tokenize_dataset(tokenizer, dsdict, target_max_len):
    def tok(batch):
        # encode inputs
        model_inputs = tokenizer(
            batch["input_text"],
            truncation=True,
            max_length=MAX_INPUT_LEN,
        )
        # encode targets (labels) separately
        labels = tokenizer(
            batch["target_text"],
            truncation=True,
            max_length=target_max_len,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    # remove original text columns to save RAM
    remove_cols = dsdict["train"].column_names
    return dsdict.map(tok, batched=True, remove_columns=remove_cols)

def run_finetune_experiment(
    model_name: str,
    view_name: str,           # "sft" or "cot"
    method: str,              # "full" or "softprompt"
    prompt_tokens: int = 30,  # for softprompt
    output_dir: str = "./runs"
):
    assert view_name in ("sft","cot")
    assert method in ("full","softprompt")

    print("\n==============================")
    print("MODEL:", model_name)
    print("VIEW:", view_name, "METHOD:", method, "m:", prompt_tokens if method=="softprompt" else None)
    print("==============================")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # soft prompt: freeze base, train only virtual tokens
    if method == "softprompt":
        peft_cfg = PromptTuningConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            prompt_tuning_init="RANDOM",
            num_virtual_tokens=prompt_tokens,
            tokenizer_name_or_path=model_name,
        )
        model = get_peft_model(model, peft_cfg)

    model.to(DEVICE)

    target_max_len = MAX_TARGET_LEN_COT if view_name=="cot" else MAX_TARGET_LEN_LABEL
    tokenized = tokenize_dataset(tokenizer, views[view_name], target_max_len=target_max_len)

    collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

    # unified training args
    run_name = f"{model_name.replace('/','_')}-{view_name}-{method}" + (f"-m{prompt_tokens}" if method=="softprompt" else "")
    out = os.path.join(output_dir, run_name)

    args = Seq2SeqTrainingArguments(
        output_dir=out,
        do_train=True,
        do_eval=True,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=(LR_PROMPT if method=="softprompt" else LR_FULL),
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        fp16=(DEVICE == "cuda"),
        # 这里不再传 generation_max_new_tokens，统一解码预算由 timed_generate 控制
        report_to=[],
        seed=SEED,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        data_collator=collator,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
    )

    # ---- train ----
    train_res = trainer.train()

    # ---- evaluate via generate (validation) ----
    val_inputs = [INSTR_TEMPLATE.format(statement=x["statement"]) for x in ds["validation"]] \
        if view_name=="sft" else [COT_TEMPLATE.format(statement=x["statement"]) for x in ds["validation"]]
    val_gold = [x["label"] for x in ds["validation"]]
    val_pred_texts, val_s_per_ex = timed_generate(model, tokenizer, val_inputs, batch_size=8)
    val_metrics = compute_metrics_from_text(val_pred_texts, val_gold)

    # ---- test ----
    test_inputs = [INSTR_TEMPLATE.format(statement=x["statement"]) for x in ds["test"]] \
        if view_name=="sft" else [COT_TEMPLATE.format(statement=x["statement"]) for x in ds["test"]]
    test_gold = [x["label"] for x in ds["test"]]
    test_pred_texts, test_s_per_ex = timed_generate(model, tokenizer, test_inputs, batch_size=8)
    test_metrics = compute_metrics_from_text(test_pred_texts, test_gold)

    total_p, trainable_p = count_params(model)
    # storage: save full or adapter
    if method == "softprompt":
        model.save_pretrained(out + "/adapter")
        saved_path = out + "/adapter"
    else:
        model.save_pretrained(out + "/full_model")
        saved_path = out + "/full_model"

    # compute folder size (MB)
    def folder_size_mb(path):
        s = 0
        for root, _, files in os.walk(path):
            for f in files:
                s += os.path.getsize(os.path.join(root, f))
        return s / (1024**2)

    size_mb = folder_size_mb(saved_path)

    result = {
        "run": run_name,
        "model": model_name,
        "view": view_name,
        "method": method,
        "m": (prompt_tokens if method=="softprompt" else None),
        "val_macro_f1": val_metrics["macro_f1"],
        "val_acc": val_metrics["acc"],
        "val_valid_rate": val_metrics["valid_rate"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_acc": test_metrics["acc"],
        "test_valid_rate": test_metrics["valid_rate"],
        "val_sec_per_ex": val_s_per_ex,
        "test_sec_per_ex": test_s_per_ex,
        "total_params": total_p,
        "trainable_params": trainable_p,
        "saved_mb": size_mb,
    }

    # cleanup
    del trainer, model
    torch.cuda.empty_cache()
    gc.collect()

    return result

In [9]:
def run_prompting_baseline(model_name: str, fewshot_k: int = 0):
    print("\n==============================")
    print("BASELINE MODEL:", model_name, "fewshot_k:", fewshot_k)
    print("==============================")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)

    # build few-shot prefix from train set (balanced)
    rng = np.random.RandomState(SEED)
    train_pairs = [(x["statement"], x["label"]) for x in ds["train"]]
    tr_true = [p for p in train_pairs if p[1]=="TRUE"]
    tr_fake = [p for p in train_pairs if p[1]=="FAKE"]

    prefix = ""
    if fewshot_k > 0:
        k_true = fewshot_k // 2
        k_fake = fewshot_k - k_true
        ex = rng.choice(len(tr_true), size=k_true, replace=False).tolist()
        ex2 = rng.choice(len(tr_fake), size=k_fake, replace=False).tolist()
        examples = [tr_true[i] for i in ex] + [tr_fake[i] for i in ex2]
        rng.shuffle(examples)

        for st, lb in examples:
            prefix += INSTR_TEMPLATE.format(statement=st) + f" {lb}\n\n"

    val_inputs = []
    for x in ds["validation"]:
        val_inputs.append(prefix + INSTR_TEMPLATE.format(statement=x["statement"]))

    val_gold = [x["label"] for x in ds["validation"]]
    val_pred_texts, val_s_per_ex = timed_generate(model, tokenizer, val_inputs, batch_size=8)
    val_metrics = compute_metrics_from_text(val_pred_texts, val_gold)

    # test
    test_inputs = []
    for x in ds["test"]:
        test_inputs.append(prefix + INSTR_TEMPLATE.format(statement=x["statement"]))
    test_gold = [x["label"] for x in ds["test"]]
    test_pred_texts, test_s_per_ex = timed_generate(model, tokenizer, test_inputs, batch_size=8)
    test_metrics = compute_metrics_from_text(test_pred_texts, test_gold)

    total_p = sum(p.numel() for p in model.parameters())
    # approximate model size by saving once (optional); to save time, we skip by default
    result = {
        "run": f"{model_name.replace('/','_')}-prompting-k{fewshot_k}",
        "model": model_name,
        "view": "sft",
        "method": f"prompting_k{fewshot_k}",
        "m": None,
        "val_macro_f1": val_metrics["macro_f1"],
        "val_acc": val_metrics["acc"],
        "val_valid_rate": val_metrics["valid_rate"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_acc": test_metrics["acc"],
        "test_valid_rate": test_metrics["valid_rate"],
        "val_sec_per_ex": val_s_per_ex,
        "test_sec_per_ex": test_s_per_ex,
        "total_params": total_p,
        "trainable_params": 0,
        "saved_mb": None,
    }

    del model
    torch.cuda.empty_cache()
    gc.collect()
    return result

In [10]:
results = []

# ========== Research Subject 1 ==========
small_models = [
    "google/flan-t5-small",
    "facebook/bart-base",
]

# Instruction SFT
for mn in small_models:
    results.append(run_finetune_experiment(mn, view_name="sft", method="full"))

# Instruction + CoT SFT
for mn in small_models:
    results.append(run_finetune_experiment(mn, view_name="cot", method="full"))

# Instruction + Soft Prompt-Tuning (m=20~50)
for mn in small_models:
    for m in [20, 30, 50]:
        results.append(run_finetune_experiment(mn, view_name="sft", method="softprompt", prompt_tokens=m))

# ========== Research Subject 2 ==========
# Scale baseline: large model prompting (zero-shot + few-shot)
scale_models = [
    ("google/flan-t5-base", [0, 4]),
    ("facebook/bart-large", [0, 4]),
]
for mn, ks in scale_models:
    for k in ks:
        results.append(run_prompting_baseline(mn, fewshot_k=k))

df = pd.DataFrame(results)
df.sort_values(["model","val_macro_f1"], ascending=[True, False]).reset_index(drop=True)


MODEL: google/flan-t5-small
VIEW: sft METHOD: full m: None


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:649: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:649: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(



BASELINE MODEL: facebook/bart-large fewshot_k: 4


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:649: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:649: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


,run,model,view,method,m,val_macro_f1,val_acc,val_valid_rate,test_macro_f1,test_acc,test_valid_rate,val_sec_per_ex,test_sec_per_ex,total_params,trainable_params,saved_mb
0,facebook_bart-base-sft-full,facebook/bart-base,sft,full,NaN,0.637414,0.644860,1.0,0.633694,0.655495,1.0,0.006853,0.006791,139420416,139420416,532.068852
1,facebook_bart-base-cot-full,facebook/bart-base,cot,full,NaN,0.626009,0.640187,1.0,0.616593,0.648480,1.0,0.025483,0.024951,139420416,139420416,532.068852
2,facebook_bart-base-sft-softprompt-m20,facebook/bart-base,sft,softprompt,20.0,0.368132,0.520249,1.0,0.386632,0.563523,1.0,0.006642,0.006508,139451136,30720,0.122601
3,facebook_bart-base-sft-softprompt-m30,facebook/bart-base,sft,softprompt,30.0,0.346926,0.517913,1.0,0.366236,0.561964,1.0,0.006816,0.006841,139466496,46080,0.181194
4,facebook_bart-base-sft-softprompt-m50,facebook/bart-base,sft,softprompt,50.0,0.342213,0.520249,1.0,0.361692,0.566641,1.0,0.007320,0.007187,139497216,76800,0.298382
5,facebook_bart-large-prompting-k0,facebook/bart-large,sft,prompting_k0,NaN,0.342213,0.520249,1.0,0.361692,0.566641,1.0,0.082367,0.082417,406291456,0,NaN
6,facebook_bart-large-prompting-k4,facebook/bart-large,sft,prompting_k4,NaN,0.342213,0.520249,1.0,0.361692,0.566641,1.0,0.113563,0.112995,406291456,0,NaN
7,google_flan-t5-base-prompting-k4,google/flan-t5-base,sft,prompting_k4,NaN,0.521328,0.532710,1.0,0.500995,0.508184,1.0,0.031743,0.031871,247577856,0,NaN
8,google_flan-t5-base-prompting-k0,google/flan-t5-base,sft,prompting_k0,NaN,0.516566,0.530374,1.0,0.500414,0.508184,1.0,0.009892,0.009856,247577856,0,NaN
9,google_flan-t5-small-sft-softprompt-m20,google/flan-t5-small,sft,softprompt,20.0,0.344009,0.521028,1.0,0.363346,0.566641,1.0,0.006316,0.006436,76981632,20480,0.083543


In [11]:
display(df[[
    "run","model","view","method","m",
    "val_macro_f1","val_acc","val_valid_rate",
    "test_macro_f1","test_acc","test_valid_rate",
    "test_sec_per_ex","trainable_params","saved_mb"
]].sort_values(["test_macro_f1"], ascending=False).head(30))

df.to_csv("all_results.csv", index=False)
print("Saved -> all_results1.csv")

,run,model,view,method,m,val_macro_f1,val_acc,val_valid_rate,test_macro_f1,test_acc,test_valid_rate,test_sec_per_ex,trainable_params,saved_mb
1,facebook_bart-base-sft-full,facebook/bart-base,sft,full,NaN,0.637414,0.644860,1.0,0.633694,0.655495,1.0,0.006791,139420416,532.068852
3,facebook_bart-base-cot-full,facebook/bart-base,cot,full,NaN,0.626009,0.640187,1.0,0.616593,0.648480,1.0,0.024951,139420416,532.068852
11,google_flan-t5-base-prompting-k4,google/flan-t5-base,sft,prompting_k4,NaN,0.521328,0.532710,1.0,0.500995,0.508184,1.0,0.031871,0,NaN
10,google_flan-t5-base-prompting-k0,google/flan-t5-base,sft,prompting_k0,NaN,0.516566,0.530374,1.0,0.500414,0.508184,1.0,0.009856,0,NaN
7,facebook_bart-base-sft-softprompt-m20,facebook/bart-base,sft,softprompt,20.0,0.368132,0.520249,1.0,0.386632,0.563523,1.0,0.006508,30720,0.122601
8,facebook_bart-base-sft-softprompt-m30,facebook/bart-base,sft,softprompt,30.0,0.346926,0.517913,1.0,0.366236,0.561964,1.0,0.006841,46080,0.181194
0,google_flan-t5-small-sft-full,google/flan-t5-small,sft,full,NaN,0.343669,0.520249,1.0,0.364012,0.564302,1.0,0.007721,76961152,293.606515
2,google_flan-t5-small-cot-full,google/flan-t5-small,cot,full,NaN,0.343329,0.519470,1.0,0.363346,0.566641,1.0,0.007588,76961152,293.606515
4,google_flan-t5-small-sft-softprompt-m20,google/flan-t5-small,sft,softprompt,20.0,0.344009,0.521028,1.0,0.363346,0.566641,1.0,0.006436,20480,0.083543
5,google_flan-t5-small-sft-softprompt-m30,google/flan-t5-small,sft,softprompt,30.0,0.342213,0.520249,1.0,0.361692,0.566641,1.0,0.006438,30720,0.122605


Saved -> all_results.csv
